In [7]:
import scipy.special
import numpy
import matplotlib.pyplot
import scipy.misc
%matplotlib inline

class neuralNetwork():
    def __init__(self,inputNodes,hiddenNodes,outputNodes,learnRate):
        self.inodes=inputNodes
        self.hnodes=hiddenNodes
        self.onodes=outputNodes

        self.lr=learnRate

        self.wih = numpy.random.normal(0.0, pow(self.hnodes,-0.5),(self.hnodes, self.inodes))
        self.who = numpy.random.normal(0.0, pow(self.onodes,-0.5),(self.onodes, self.hnodes))

        self.inverse_activation_function = lambda x: scipy.special.logit(x)
        self.activation_function = lambda x: scipy.special.expit(x)
        pass
        
    def train(self, inputs_list, targets_list):
        inputs=numpy.array(inputs_list, ndmin=2).T
        targets = numpy.array(targets_list, ndmin=2).T

        hidden_inputs = numpy.dot(self.wih, inputs)
        hidden_outputs = self.activation_function(hidden_inputs)
        
        final_inputs = numpy.dot(self.who, hidden_outputs)
        final_outputs = self.activation_function(final_inputs)

        output_errors = targets - final_outputs

        hidden_errors=numpy.dot(self.who.T,output_errors)

        self.who += self.lr * numpy.dot((output_errors * final_outputs * (1.0 - final_outputs)), numpy.transpose(hidden_outputs))

        self.wih+= self.lr * numpy.dot((hidden_errors* hidden_outputs * (1.0 - hidden_outputs)), numpy.transpose(inputs))
        pass

    def query(self, inputs_list):
        inputs = numpy.array(inputs_list, ndmin=2).T

        hidden_inputs = numpy.dot(self.wih, inputs)
        hidden_outputs = self.activation_function(hidden_inputs)
        
        final_inputs = numpy.dot(self.who, hidden_outputs)
        final_outputs = self.activation_function(final_inputs)
        return final_outputs

    def backquery(self, targets_list):
        final_outputs = numpy.array(targets_list, ndmin=2).T
        
        final_inputs = self.inverse_activation_function(final_outputs)

        hidden_outputs = numpy.dot(self.who.T, final_inputs)
        hidden_outputs -= numpy.min(hidden_outputs)
        hidden_outputs /= numpy.max(hidden_outputs)
        hidden_outputs *= 0.98
        hidden_outputs += 0.01
        
        hidden_inputs = self.inverse_activation_function(hidden_outputs)
        
        inputs = numpy.dot(self.wih.T, hidden_inputs)
        inputs -= numpy.min(inputs)
        inputs /= numpy.max(inputs)
        inputs *= 0.98
        inputs += 0.01
        
        return inputs

input_nodes = 784
hidden_nodes = 200
output_nodes = 10

learning_rate = 0.05

n = neuralNetwork(input_nodes,hidden_nodes,output_nodes,learning_rate)

data_file = open("Downloads/mnist_train.csv",'r')
data_list = data_file.readlines()
data_file.close()

epoch=5

for j in range(epoch):
    for i in data_list:
        all_values = i.split(',')
        scaled_input = (numpy.asarray(all_values[1:],dtype=float) / 255.0 * 0.99) + 0.01
        image_rot=scipy.ndimage.interpolation.rotate(scaled_input.reshape(28,28), (j*5)-10,cval=0.01, reshape=False)
        image_rot_flat = image_rot.flatten()
        targets = numpy.zeros(output_nodes) + 0.01 
        targets[int(all_values[0])] = 0.99
        n.train(image_rot_flat, targets)
        pass
pass

data_file1 = open("Downloads/mnist_test.csv",'r')
data_list1 = data_file1.readlines()
data_file1.close()
scoreboard=[]

for i in data_list1:
    all_values = i.split(',')
    correct_label = int(all_values[0])
    #print("Correct label - ",correct_label)
    scaled_input = (numpy.asarray(all_values[1:],dtype=float) / 255.0 * 0.99) + 0.01
    outputs = n.query(scaled_input)
    label = numpy.argmax(outputs)
    #print("Network response - ",label)
    if(label==correct_label):
        scoreboard.append(1)
    else:
        scoreboard.append(0)
    
#print(scoreboard)
scoreboard_array = numpy.asarray(scoreboard)
print ("performance = ", scoreboard_array.sum() / scoreboard_array.size)

'''
label = 9
targets = numpy.zeros(output_nodes) + 0.01
targets[label] = 0.99
print(targets)

image_data = n.backquery(targets)

matplotlib.pyplot.imshow(image_data.reshape(28,28), cmap='Greys', interpolation='None')
'''

C:\Users\Ihor\AppData\Local\Temp\ipykernel_30392\1380231209.py:90: DeprecationWarning: Please import `rotate` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  image_rot=scipy.ndimage.interpolation.rotate(scaled_input.reshape(28,28), (j*5)-10,cval=0.01, reshape=False)


performance =  0.9672


"\nlabel = 9\ntargets = numpy.zeros(output_nodes) + 0.01\ntargets[label] = 0.99\nprint(targets)\n\nimage_data = n.backquery(targets)\n\nmatplotlib.pyplot.imshow(image_data.reshape(28,28), cmap='Greys', interpolation='None')\n"

In [16]:
import scipy.special
import numpy
import matplotlib.pyplot
import scipy.misc
%matplotlib inline

class neuralNetwork():
    def __init__(self,inputNodes,hiddenNodes,outputNodes,learnRate):
        self.inodes=inputNodes
        self.hnodes=hiddenNodes
        self.onodes=outputNodes

        self.lr=learnRate

        self.wih = numpy.random.randn(self.hnodes, self.inodes) * numpy.sqrt(2.0/self.inodes)
        self.who = numpy.random.randn(self.onodes, self.hnodes) * numpy.sqrt(2.0/self.hnodes)

        #self.inverse_activation_function = lambda x: scipy.special.logit(x)
        pass

    def activation_function(self, x):
        return numpy.maximum(0, x)
            
    def train(self, inputs_list, targets_list):
        inputs=numpy.array(inputs_list, ndmin=2).T
        targets = numpy.array(targets_list, ndmin=2).T

        hidden_inputs = numpy.dot(self.wih, inputs)
        hidden_outputs = self.activation_function(hidden_inputs)
        
        final_inputs = numpy.dot(self.who, hidden_outputs)
        final_outputs = self.activation_function(final_inputs)

        output_errors = targets - final_outputs

        hidden_errors=numpy.dot(self.who.T,output_errors)

        output_grad = output_errors * (final_inputs > 0).astype(float)
        self.who += self.lr * numpy.dot(output_grad, numpy.transpose(hidden_outputs))

        hidden_grad = hidden_errors * (hidden_inputs > 0).astype(float)
        self.wih+= self.lr * numpy.dot(hidden_grad, numpy.transpose(inputs))
        pass

    def query(self, inputs_list):
        inputs = numpy.array(inputs_list, ndmin=2).T

        hidden_inputs = numpy.dot(self.wih, inputs)
        hidden_outputs = self.activation_function(hidden_inputs)
        
        final_inputs = numpy.dot(self.who, hidden_outputs)
        final_outputs = self.activation_function(final_inputs)
        return final_outputs
'''
    def backquery(self, targets_list):
        final_outputs = numpy.array(targets_list, ndmin=2).T
        
        final_inputs = self.inverse_activation_function(final_outputs)

        hidden_outputs = numpy.dot(self.who.T, final_inputs)
        hidden_outputs -= numpy.min(hidden_outputs)
        hidden_outputs /= numpy.max(hidden_outputs)
        hidden_outputs *= 0.98
        hidden_outputs += 0.01
        
        hidden_inputs = self.inverse_activation_function(hidden_outputs)
        
        inputs = numpy.dot(self.wih.T, hidden_inputs)
        inputs -= numpy.min(inputs)
        inputs /= numpy.max(inputs)
        inputs *= 0.98
        inputs += 0.01
        
        return inputs
        '''

input_nodes = 784
hidden_nodes = 256
output_nodes = 10

learning_rate = 0.001

n = neuralNetwork(input_nodes,hidden_nodes,output_nodes,learning_rate)

data_file = open("Downloads/mnist_train.csv",'r')
data_list = data_file.readlines()
data_file.close()

epoch=5

for j in range(epoch):
    for i in data_list:
        all_values = i.split(',')
        scaled_input = (numpy.asarray(all_values[1:],dtype=float) / 255.0 * 1.99) - 1
        targets = numpy.zeros(output_nodes) 
        targets[int(all_values[0])] = 0.99
        n.train(scaled_input, targets)
        pass
pass

data_file1 = open("Downloads/mnist_test.csv",'r')
data_list1 = data_file1.readlines()
data_file1.close()
scoreboard=[]

for i in data_list1:
    all_values = i.split(',')
    correct_label = int(all_values[0])
    #print("Correct label - ",correct_label)
    scaled_input = (numpy.asarray(all_values[1:],dtype=float) / 255.0 * 1.99) -1
    outputs = n.query(scaled_input)
    label = numpy.argmax(outputs)
    #print("Network response - ",label)
    if(label==correct_label):
        scoreboard.append(1)
    else:
        scoreboard.append(0)
    
#print(scoreboard)
scoreboard_array = numpy.asarray(scoreboard)
print ("performance = ", scoreboard_array.sum() / scoreboard_array.size)

'''
label = 9
targets = numpy.zeros(output_nodes) + 0.01
targets[label] = 0.99
print(targets)

image_data = n.backquery(targets)

matplotlib.pyplot.imshow(image_data.reshape(28,28), cmap='Greys', interpolation='None')
'''

performance =  0.9637


"\nlabel = 9\ntargets = numpy.zeros(output_nodes) + 0.01\ntargets[label] = 0.99\nprint(targets)\n\nimage_data = n.backquery(targets)\n\nmatplotlib.pyplot.imshow(image_data.reshape(28,28), cmap='Greys', interpolation='None')\n"